Of course. Here is a complete, cell-by-cell explanation of your fine-tuning program formatted in Markdown, ready to be added to a Jupyter Notebook.

-----

# **Fine-Tuning DeepSeek Coder 6.7B with QLoRA for Finacle Script and MRT**

This notebook demonstrates how to efficiently fine-tune the `deepseek-coder-6.7b-instruct` model on a custom dataset. We'll use **QLoRA** (Quantized Low-Rank Adaptation), a technique that makes it possible to train large models on a single consumer-grade GPU.

QLoRA combines two key methods:

  * **Quantization**: Reduces the model's memory footprint by loading it in 4-bit precision.
  * **LoRA (Low-Rank Adaptation)**: Freezes the large base model and only trains small, efficient "adapter" layers, drastically reducing the number of trainable parameters.

-----

### **Step 1: Setup and Imports**

First, we import the necessary libraries for data handling, model loading, and training from PyTorch, Transformers, and PEFT.

In [1]:
!pip install torch transformers peft bitsandbytes accelerate matplotlib numpy ipython
import json
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
)
from peft import get_peft_model, LoraConfig, TaskType
import os

# For plotting
import matplotlib.pyplot as plt
from IPython.display import clear_output
from transformers import TrainerCallback
import numpy as np
import time

print("library loaded")

library loaded


### **Step 1.1: Initialize parameters**

Initialize the value for maximum token length and model name

In [2]:
import ipywidgets as widgets
from IPython.display import display

# Create a text box widget for the model name
token_max_length = widgets.IntSlider(
    value=512,
    min=256,
    max=2056,
    step=256,
    description='Token Length:'
)

model_widget = widgets.Text(
    value='deepseek-ai/deepseek-coder-6.7b-instruct',
    description='Model Name:',
    style={'description_width': 'initial'}
)


# Display the widgets
display(model_widget, token_max_length)

Text(value='deepseek-ai/deepseek-coder-6.7b-instruct', description='Model Name:', style=DescriptionStyle(descr…

IntSlider(value=512, description='Token Length:', max=2056, min=256, step=256)

### **Step 1.2: Define calss for plotting the training data**

Initialize the value for maximum token length and model name

In [3]:
class AdvancedPlottingCallback(TrainerCallback):
    """
    A custom callback that plots training loss, learning rate, and progress metrics in real-time.
    """
    def __init__(self, smoothing_window=10):
        super().__init__()
        self.steps = []
        self.losses = []
        self.learning_rates = []
        self.smoothing_window = smoothing_window
        self.last_log_time = time.time()

    def _smooth(self, values):
        """Applies a simple moving average to a list of values."""
        if len(values) < self.smoothing_window:
            return values
        return np.convolve(values, np.ones(self.smoothing_window)/self.smoothing_window, mode='valid').tolist()

    def _format_time(self, seconds):
        """Formats seconds into HH:MM:SS."""
        if seconds is None or seconds < 0:
            return "N/A"
        m, s = divmod(seconds, 60)
        h, m = divmod(m, 60)
        return f"{int(h):02d}:{int(m):02d}:{int(s):02d}"

    def on_log(self, args, state, control, logs=None, **kwargs):
        # --- THE FIX: Only proceed if 'loss' is in the logs ---
        # This filters out the final summary log that doesn't have step-wise data.
        if logs is None or 'loss' not in logs:
            return

        # --- Performance Metrics Calculation ---
        current_time = time.time()
        time_since_last_log = current_time - self.last_log_time

        last_step = self.steps[-1] if self.steps else 0
        steps_since_last_log = state.global_step - last_step
        steps_per_second = 0
        if time_since_last_log > 0 and steps_since_last_log > 0:
            steps_per_second = steps_since_last_log / time_since_last_log

        remaining_steps = state.max_steps - state.global_step
        eta_seconds = remaining_steps / steps_per_second if steps_per_second > 0 else None

        self.last_log_time = current_time

        # --- Append data for plotting ---
        self.steps.append(state.global_step)
        self.losses.append(logs['loss'])
        if 'learning_rate' in logs:
            self.learning_rates.append(logs['learning_rate'])

        if len(self.steps) < 2:
            return

        # --- Plotting ---
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

        # --- Main Title with Progress Info ---
        progress_str = f"Step: {state.global_step}/{state.max_steps}"
        epoch_str = f"Epoch: {state.epoch:.2f}"
        rate_str = f"{steps_per_second:.2f} it/s"
        eta_str = f"ETA: {self._format_time(eta_seconds)}"
        fig.suptitle(f"Training Progress | {progress_str} | {epoch_str} | {rate_str} | {eta_str}", fontsize=16)

        # --- Plot 1: Training Loss ---
        ax1.plot(self.steps, self.losses, color='grey', alpha=0.5, label='Raw Loss')
        smoothed_losses = self._smooth(self.losses)
        smoothed_steps = self.steps[self.smoothing_window - 1:]
        if smoothed_steps:
             ax1.plot(smoothed_steps, smoothed_losses, color='blue', label=f'Smoothed Loss (window={self.smoothing_window})')
        ax1.set_title("Training Loss")
        ax1.set_ylabel("Loss")
        ax1.grid(True)
        ax1.legend()

        # --- Plot 2: Learning Rate ---
        ax2.plot(self.steps, self.learning_rates, color='green', marker='.')
        ax2.set_title("Learning Rate Schedule")
        ax2.set_xlabel("Steps")
        ax2.set_ylabel("Learning Rate")
        ax2.grid(True)

        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()

### **Step 1.3: Check the availability of cuda**

Initialize the value for maximum token length and model name

In [4]:
# --- Check for GPU availability ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"Device name: {torch.cuda.get_device_name(0)}")
# ------------------------------------

Using device: cuda
Device name: Tesla T4


-----

### **Step 2: Create a Custom Dataset Class**

We need a way to load our data and prepare it for the model. The `FinacleScriptDataset` class handles this. It scans a directory for all `.json` files, reads them, and formats each entry into a single string: `PROMPT: ...\nSCRIPT: ...<|endoftext|>`. This structured format, ending with an end-of-sequence token, teaches the model how to respond to prompts. Finally, it uses the tokenizer to convert this text into numbers that the model can process.

In [5]:
class FinacleScriptDataset(Dataset):
    def __init__(self, directory_path, tokenizer):
        self.tokenizer = tokenizer
        self.examples = []

        print(f"Loading all .json files from directory: {directory_path}")
        print(f'token length = "{token_max_length.value}", model used is "{model_widget.value}"')

        # Check if the directory exists
        if not os.path.isdir(directory_path):
            raise ValueError(f"Provided path '{directory_path}' is not a directory.")

        # Loop through all files in the specified directory
        for filename in os.listdir(directory_path):
            if filename.endswith(".json"):
                file_path = os.path.join(directory_path, filename)
                print(f"  - Loading data from: {filename}")
                with open(file_path, "r", encoding="utf-8") as f:
                    data = json.load(f)
                    for item in data:
                        # The text formatting remains the same
                        text = f"PROMPT: {item['prompt']}\nSCRIPT: {item['script']}{self.tokenizer.eos_token}"
                        tokenized_text = self.tokenizer(
                            text, truncation=True, max_length=token_max_length.value
                        )
                        self.examples.append(tokenized_text)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return self.examples[i]

-----

### **Step 3: Configure 4-bit Quantization (The "Q" in QLoRA)**

To fit the 6.7 billion parameter model into memory, we use 4-bit quantization. The `BitsAndBytesConfig` object allows us to specify exactly how the model should be loaded, using Normal Float 4 (`nf4`) precision for storage while performing the actual computations in 16-bit (`bfloat16`) for accuracy.

In [6]:
print("Configuring 4-bit quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
print("Done.")

Configuring 4-bit quantization...
Done.


-----

### **Step 4: Load the Base Model and Tokenizer**

Now we load the pre-trained model and its tokenizer from the Hugging Face Hub. The crucial `quantization_config` argument applies our 4-bit configuration, and `device_map="auto"` automatically places the model on the available GPU. We also set the tokenizer's padding token, a standard practice for these models.

In [ ]:
print("Loading base model and tokenizer with 4-bit quantization...")
model_name = model_widget.value
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Done.")

Loading base model and tokenizer with 4-bit quantization...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

-----

### **Step 5: Configure and Apply LoRA (The "LoRA" in QLoRA)**

With the base model loaded but frozen, we now configure the LoRA adapters. These are the only parts of the model that will be trained. We target the attention mechanism's projection layers (`q_proj`, `k_proj`, etc.), as this is highly effective. The `get_peft_model` function wraps our quantized model with these adapters. As you can see from the output of `print_trainable_parameters()`, we will only be training a tiny fraction of the total parameters.

In [ ]:
print("Configuring LoRA adapter...")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("Done.")

-----

### **Step 6: Prepare Dataset and Training Arguments**

Here, we instantiate our custom dataset and define the training arguments. `TrainingArguments` is an object that holds all the hyperparameters for the training run, such as the number of epochs, batch size, learning rate, and the memory-efficient `paged_adamw_8bit` optimizer.

In [ ]:
print("Preparing dataset...")
# <<<< Define the directory containing your training files
train_directory_path = "."
# <<<< Instantiate the dataset with the directory path
train_dataset = FinacleScriptDataset(
    directory_path=train_directory_path, tokenizer=tokenizer
)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print(f"Dataset prepared with {len(train_dataset)} total examples.")
print("Done.")

# --- 6. Training Arguments (No changes) ---
training_args = TrainingArguments(
    output_dir="./qlora-finetuned-deepseek-7b",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=10,
    logging_dir="./logs",
    logging_steps=1,
    fp16=True,
    optim="paged_adamw_8bit",
)

-----

### **Step 7: Initialize Trainer and Start Training**

With all components prepared, we initialize the `Trainer`, which abstracts away the entire training loop. The `.train()` command kicks off the fine-tuning process.

In [ ]:
# 1. Instantiate the new multi-metric callback
# You can adjust the smoothing_window for the loss curve
plotting_callback = AdvancedPlottingCallback(smoothing_window=10)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    callbacks=[plotting_callback], # <--- Use the new callback
)

print("Starting QLoRA fine-tuning on DeepSeek Coder 6.7B...")
trainer.train()
print("Training complete!")


-----

### **Step 8: Save the Final Adapter**

After training, we save our work. Crucially, we are **not** saving the entire 6.7B parameter model. We are only saving the small, lightweight LoRA adapter we just trained. This adapter can then be loaded on top of the original base model for inference later.

In [ ]:
print("Saving the LoRA adapter...")
model.save_pretrained("./qlora-finetuned-deepseek-7b")
tokenizer.save_pretrained("./qlora-finetuned-deepseek-7b")
print("Model and tokenizer saved.")